In [4]:
from huggingface_hub import HfApi, login
from huggingface_hub import create_repo

with open('/home/jovyan/kurkin/hf_key_read.txt', 'r') as file:
    hf_key = file.read().strip()

login(hf_key)

In [1]:
from openai import OpenAI, AsyncOpenAI

client = AsyncOpenAI(api_key='YOUR_API_KEY', base_url='http://0.0.0.0:8000/v1')

In [2]:
model_names = await client.models.list()
model_name = model_names.data[0].id
model_name

'microsoft/Phi-3.5-vision-instruct'

In [3]:
! pwd

/home/jovyan/shares/SR004.nfs2/kurkin/omni/long-vqa


In [4]:
messages = [{
        'role': 'user',
        'content': [
            {
                'type': 'text',
                'text': 'You are an assistant that analyzes image sequences. Answer with one word. Where was Mary last time?',
            }, 
            {
                'type': 'image_url',
                'image_url': {
                    'url':
                    'file:///home/jovyan/shares/SR004.nfs2/kurkin/omni/long-vqa/data/length_256/sequence_000/step_000.png',
                },
            },
            {
                'type': 'image_url',
                'image_url': {
                    'url':
                    'file:///home/jovyan/shares/SR004.nfs2/kurkin/omni/long-vqa/data/length_256/sequence_000/step_001.png',
                },
            },
        ],
    }]

In [6]:
# import asyncio

# responses = await asyncio.gather(*[
#         client.chat.completions.create(
#             model=model_name,
#             messages=messages,
#             temperature=0.0,
#             top_p=0.8) 
#         for _ in range(32)
#     ])

In [5]:
response = await client.chat.completions.create(
    model=model_name,
    messages=messages,
    temperature=0.0,
    top_p=0.8)
print(response)

ChatCompletion(id='chatcmpl-3dbb2460b39d4277b39f4bfad8d00201', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content=' Hallway', refusal=None, role='assistant', audio=None, function_call=None, tool_calls=[]), stop_reason=None)], created=1735310970, model='microsoft/Phi-3.5-vision-instruct', object='chat.completion', service_tier=None, system_fingerprint=None, usage=CompletionUsage(completion_tokens=4, prompt_tokens=1549, total_tokens=1553, completion_tokens_details=None, prompt_tokens_details=None), prompt_logprobs=None)


In [6]:
len(response.choices)

1

In [7]:
print(response.choices[0].message.content)

 Hallway


In [14]:
import asyncio
import csv
import os
import glob
from typing import List, Tuple
import openai
from tqdm.asyncio import tqdm_asyncio
import pandas as pd

def read_csv(file_path: str) -> List[Tuple[str, str, str]]:
    df = pd.read_csv(QA_PAIRS_CSV).sort_values("N_steps")
    return df.to_dict("records")

def get_image_urls(qa: dict, current_dir: str) -> List[dict]:
    DATA_DIR = 'data/length_256'
    sequence_dir = os.path.join(DATA_DIR, qa['Seq_id'])
    image_pattern = os.path.join(sequence_dir, 'step_*.png')

    image_paths = sorted(glob.glob(image_pattern))

    if len(image_paths):
        image_paths = image_paths[:int(qa['N_steps'])]

    image_urls = []
    for path in image_paths:
        # relative_path = os.path.relpath(path, current_dir)
        # file_url = f'{relative_path.replace(os.sep, "/")}'
        
        image_urls.append({'image_url': {'url': "file:///" + path}})
    return image_urls

async def process_qa_pair(qa: dict, current_dir: str, semaphore: asyncio.Semaphore) -> Tuple[dict, str]:
    image_urls = get_image_urls(qa, current_dir)
    if not image_urls:
        return (qa, "No images found for the given sequence.")
    user_content = [{'type': 'text', 'text': qa['Question']}]
    user_content += [url | {"type": "image_url"} for url in image_urls]

    # Prepare messages
    messages = [
        {"role": "system", "content": "You are an assistant that analyzes image sequences. Answer with one word or number (digit 0-9)."},
        {"role": "user", "content": user_content}
    ]

    async with semaphore:
        try:
            response = await client.chat.completions.create(
                model=model_name,
                messages=messages,
                temperature=0.0
            )
            answer = response.choices[0].message.content
            return (qa, answer)
        except Exception as e:
            return (qa, f"Error: {e}")

async def process_all_qa_pairs(qa_pairs: List[dict], current_dir: str, output_csv: str, semaphore_limit: int = 10):
    semaphore = asyncio.Semaphore(semaphore_limit)

    with open(output_csv, mode='a+', encoding='utf-8', newline='') as f_out:
        fieldnames = list(qa_pairs[0].keys()) + ['Predicted_Answer']
        writer = csv.DictWriter(f_out, fieldnames=fieldnames)
        if not os.path.getsize(output_csv):
            writer.writeheader()

    tasks = []
    for qa_pair in qa_pairs:
        task = asyncio.create_task(process_qa_pair(qa_pair, current_dir, semaphore))
        tasks.append(task)

    for coro in tqdm_asyncio.as_completed(tasks, total=len(tasks), desc="Processing QA Pairs"):
        qa_data, predicted_answer = await coro
        with open(output_csv, mode='a+', encoding='utf-8', newline='') as f_out:
            writer = csv.DictWriter(f_out, fieldnames=fieldnames)
            writer.writerow(qa_data | {'Predicted_Answer': predicted_answer})

def get_completed_seq_ids(output_csv: str) -> List[str]:
    completed_seq_ids = []
    if not os.path.exists(output_csv):
        return completed_seq_ids
    with open(output_csv, mode='r', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            if 'error' not in row['Predicted_Answer'].lower():
                completed_seq_ids.append(f"{row['Seq_id']}/{row['N_steps']}/{row['Type']}")
    return completed_seq_ids

In [9]:
# Define paths
QA_PAIRS_CSV = os.path.join('data', 'test_data.csv')
OUTPUT_CSV = os.path.join('data', f'qa_pairs_answers_{"_".join(model_name.split("/"))}.csv')
CURRENT_DIR = os.getcwd()

# Step 1: Read all QA pairs synchronously
qa_pairs = read_csv(QA_PAIRS_CSV)
print(f"Total QA pairs to process: {len(qa_pairs)}")

# Step 2: Check for already completed QA pairs and filter them out
completed_seq_ids = get_completed_seq_ids(OUTPUT_CSV)
print(f"Already processed {len(completed_seq_ids)} QA pairs.")

Total QA pairs to process: 5600
Already processed 0 QA pairs.


In [10]:
qa_pairs_remaining = [pair for pair in qa_pairs if f"{pair['Seq_id']}/{pair['N_steps']}/{pair['Type']}" not in completed_seq_ids]
print(f"QA pairs remaining to process: {len(qa_pairs_remaining)}")

QA pairs remaining to process: 5600


In [11]:
qa_pairs_remaining= qa_pairs_remaining[:4]

In [15]:
# Step 3: Process remaining QA pairs asynchronously with intermediate storage
if qa_pairs_remaining:
    await process_all_qa_pairs(qa_pairs_remaining, CURRENT_DIR, OUTPUT_CSV, semaphore_limit=32)
    print(f"Processing complete. Answers have been written to {OUTPUT_CSV}")
else:
    print("No QA pairs remaining to process.")


Processing QA Pairs: 100%|██████████| 4/4 [00:00<00:00, 106.45it/s]

Processing complete. Answers have been written to data/qa_pairs_answers_microsoft_Phi-3.5-vision-instruct.csv


In [25]:
OUTPUT_CSV

'data/qa_pairs_answers_unsloth_Llama-3.2-11B-Vision-Instruct.csv'

In [16]:
! cat data/qa_pairs_answers_microsoft_Phi-3.5-vision-instruct.csv

Seq_id,N_steps,Type,Question,Answer,Predicted_Answer
sequence_073,1,before_last_move,What room was John in before they went to garden for the last time?,bathroom,"Error: Error code: 400 - {'object': 'error', 'message': ""Invalid 'image_url': The file path /data/length_256/sequence_073/step_000.png must be a subpath of '--allowed-local-media-path' '/home/jovyan'."", 'type': 'BadRequestError', 'param': None, 'code': 400}"
sequence_000,1,before_last_move,What room was Michael in before they went to hallway for the last time?,bathroom,"Error: Error code: 400 - {'object': 'error', 'message': ""Invalid 'image_url': The file path /data/length_256/sequence_000/step_000.png must be a subpath of '--allowed-local-media-path' '/home/jovyan'."", 'type': 'BadRequestError', 'param': None, 'code': 400}"
sequence_073,1,when_y_comes_first,What room was Michael in when Daniel came to bedroom for the first time?,bathroom,"Error: Error code: 400 - {'object': 'error', 'message': ""Invalid 'image_url': The f

In [1]:
! rm data/qa_pairs_answers_llava-hf_llava-onevision-qwen2-7b-ov-hf.csv